# Import

In [1]:
!pip install torch

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Input

In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Fetch the Data (Vinamilk on the Ho Chi Minh Stock Exchange)
print("Downloading VNM market data...")
ticker = "VNM.VN"
# Pulling roughly 5 years of daily data to ensure we have enough sequences
df = yf.download(ticker, start="2020-01-01", end="2026-06-01")

# Flatten the MultiIndex columns from yfinance, taking the first level
# This transforms columns like ('Close', 'VNM.HM') to 'Close'
df.columns = df.columns.get_level_values(0)

# 2. The Critical Transformation: Log Returns
# We use the adjusted close to account for any stock splits or dividends
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# 3. Feature Engineering (Providing Market Context)
# Volatility: 20-day rolling standard deviation of our log returns
df['Volatility_20d'] = df['Log_Return'].rolling(window=20).std()

# Trend Distance: How far is today's price from the 50-day moving average?
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['Dist_from_SMA'] = (df['Close'] - df['SMA_50']) / df['SMA_50']

# Momentum: Simple 14-day rate of change
df['Momentum_14d'] = df['Close'] / df['Close'].shift(14) - 1

# 4. Target Creation and Cleanup
# We want to predict TOMORROW'S log return, so we shift the column backwards
df['Target_Next_Day'] = df['Log_Return'].shift(-1)

# Drop all NaN rows created by the rolling windows (the first 50 days) and the final shifted day
df = df.dropna()

# 5. Final Array Extraction
# We only pass our stationary features to the model, NEVER raw prices.
feature_cols = ['Log_Return', 'Volatility_20d', 'Dist_from_SMA', 'Momentum_14d', 'Volume']

y_raw = df[['Target_Next_Day']].values
X_raw = df[feature_cols].values

print(f"Data ready. Total valid trading days: {len(X_raw)}")
print(f"Number of engineered features: {X_raw.shape[1]}")

/tmp/ipykernel_1840/2198880327.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2020-01-01", end="2026-06-01")
[*********************100%***********************]  1 of 1 completed

Data ready. Total valid trading days: 1613
Number of engineered features: 5


In [4]:
df

Price,Close,High,Low,Open,Volume,Log_Return,Volatility_20d,SMA_50,Dist_from_SMA,Momentum_14d,Target_Next_Day
Date,,,,,,,,,,,
2020-03-11,62012.390625,64087.610393,60547.530959,62256.535810,2750892,-0.000984,0.023950,68195.301328,-0.090665,-0.045112,-0.006914
2020-03-12,61585.117188,61951.331946,59204.721255,59326.790934,2917920,-0.006914,0.023960,68004.869453,-0.094401,-0.051692,-0.018000
2020-03-13,60486.484375,61402.021446,58106.087991,58106.087991,2590956,-0.018000,0.024192,67775.375078,-0.107545,-0.086636,-0.021419
2020-03-16,59204.726562,60547.510317,58899.543755,60486.478333,1304496,-0.021419,0.024508,67515.362266,-0.123093,-0.088346,-0.010362
2020-03-17,58594.394531,58838.539749,57984.034348,58594.394531,2145888,-0.010362,0.024338,67254.130000,-0.128761,-0.111111,-0.010472
...,...,...,...,...,...,...,...,...,...,...,...
2026-05-22,59500.000000,59700.000000,58700.000000,59100.000000,6005000,0.008439,0.007594,61014.000000,-0.024814,-0.022989,-0.006745
2026-05-25,59100.000000,59800.000000,59100.000000,59500.000000,1587500,-0.006745,0.007670,60960.000000,-0.030512,-0.032733,0.000000
2026-05-26,59100.000000,59400.000000,59000.000000,59100.000000,1880800,0.000000,0.007674,60912.000000,-0.029748,-0.039024,-0.005089


# Clean and Scale the Data

In [5]:
# 3. Create TWO separate scalers
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_scaled = feature_scaler.fit_transform(X_raw)
y_scaled = target_scaler.fit_transform(y_raw)

## data preparation for Pytorch

In [6]:
# --- Updated Dataset Class ---
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback_steps, forecast_horizon=1):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.lookback = lookback_steps
        self.horizon = forecast_horizon

    def __len__(self):
        return len(self.X) - self.lookback - self.horizon + 1

    def __getitem__(self, index):
        # Window of past features: [Sequence Length, Features]
        x_window = self.X[index : index + self.lookback]

        # Target value in the future: [1]
        target_index = index + self.lookback + self.horizon - 1
        y_target = self.y[target_index]

        return x_window, y_target

In [7]:
# --- 1. Calculate the Split Indices ---
# We will use a standard chronological split: 80% Train, 10% Validation, 10% Test
total_samples = len(X_scaled)
train_end = int(total_samples * 0.8)
val_end = int(total_samples * 0.9)

# --- 2. Chronologically Slice the Arrays ---
X_train = X_scaled[:train_end]
y_train = y_scaled[:train_end]

X_val = X_scaled[train_end:val_end]
y_val = y_scaled[train_end:val_end]

X_test = X_scaled[val_end:]
y_test = y_scaled[val_end:]

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Testing samples: {len(X_test)}")

# --- 3. Instantiate the Three Datasets ---
lookback = 24

train_dataset = TimeSeriesDataset(X_train, y_train, lookback_steps=lookback)
val_dataset   = TimeSeriesDataset(X_val, y_val, lookback_steps=lookback)
test_dataset  = TimeSeriesDataset(X_test, y_test, lookback_steps=lookback)

# --- 4. Create the DataLoaders ---
# NOTE: It is safe (and often beneficial) to shuffle the sliding windows
# in the TRAINING loader only, to break up epoch-to-epoch correlations.
# NEVER shuffle the validation or test loaders.
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

Training samples: 1290
Validation samples: 161
Testing samples: 162


In [8]:
# Grab one batch of data
for X_batch, y_batch in train_dataloader:
    print(f"Input X shape:  {X_batch.shape}")
    print(f"Target y shape: {y_batch.shape}")
    break

Input X shape:  torch.Size([32, 24, 5])
Target y shape: torch.Size([32, 1])


# Deep Learning models:

## DNN

### architecture

In [9]:
class BaselineDNN(nn.Module):
    def __init__(self, lookback, features):
        super().__init__()
        # 1. The flattener
        self.flatten = nn.Flatten()

        # 2. The network stack
        # Input dimension is 24 * 13 = 312
        self.net = nn.Sequential(
            nn.Linear(lookback * features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Output a single prediction
        )

    def forward(self, x):
        # Input 'x' shape: [32, 24, 13]

        x = self.flatten(x)
        # Shape is now: [32, 312]

        predictions = self.net(x)
        # Shape is now: [32, 1]

        return predictions

### training & Dianogsis

In [10]:
# 1. Hyperparameters & Model Setup
epochs = 15
learning_rate = 0.001

# Instantiate the model with your lookback (24) and features (13)
model = BaselineDNN(lookback=24, features=len(feature_cols))

# Loss function and Optimizer
criterion = nn.MSELoss()  # MSE is perfect for regression
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Starting Training Loop...\n")

# 2. The Core Training & Validation Loop
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()  # Set model to training mode
    train_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        # X_batch shape: [32, 24, 13], y_batch shape: [32, 1]

        # Forward pass: Compute predicted y by passing x to the model
        predictions = model(X_batch)

        # Compute loss
        loss = criterion(predictions, y_batch)

        # Zero gradients, perform a backward pass, and update the weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        train_loss += loss.item() * X_batch.size(0)

    # Calculate average training loss for this epoch
    epoch_train_loss = train_loss / len(train_dataloader.dataset)

    # --- VALIDATION PHASE ---
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0

    with torch.no_grad():  # Turn off gradient computation for validation
        for X_batch, y_batch in val_dataloader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = val_loss / len(val_dataloader.dataset)

    # Print metrics every epoch
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss (MSE): {epoch_train_loss:.5f} | Val Loss (MSE): {epoch_val_loss:.5f}")


# 3.  (Evaluation on Unseen Data)
print("\nTraining Complete. Evaluating on Test Set...")
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        predictions = model(X_batch)

        # Collect predictions and actuals as numpy arrays
        all_predictions.append(predictions.numpy())
        all_actuals.append(y_batch.numpy())

# Stack all batches into single arrays
all_predictions = np.vstack(all_predictions)
all_actuals = np.vstack(all_actuals)

# 4. Post-Processing: Convert Scaled Z-Scores back to Degrees Celsius
# This uses the target_scaler we created during the data cleaning step
true_celsius = target_scaler.inverse_transform(all_actuals)
pred_celsius = target_scaler.inverse_transform(all_predictions)

# 5. Calculate Final Real-World Accuracy Metrics
final_mae = mean_absolute_error(true_celsius, pred_celsius)
final_mse = mean_squared_error(true_celsius, pred_celsius)
final_rmse = np.sqrt(final_mse)

print("\n=========================================")
print("          FINAL TEST RESULTS             ")
print("=========================================")
print(f"Mean Absolute Error (MAE):    {final_mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f}")
print("=========================================")

Starting Training Loop...

Epoch 01/15 | Train Loss (MSE): 0.83724 | Val Loss (MSE): 1.10489
Epoch 02/15 | Train Loss (MSE): 0.81085 | Val Loss (MSE): 1.09795
Epoch 03/15 | Train Loss (MSE): 0.78931 | Val Loss (MSE): 1.10531
Epoch 04/15 | Train Loss (MSE): 0.77091 | Val Loss (MSE): 1.10135
Epoch 05/15 | Train Loss (MSE): 0.74703 | Val Loss (MSE): 1.10553
Epoch 06/15 | Train Loss (MSE): 0.72340 | Val Loss (MSE): 1.13109
Epoch 07/15 | Train Loss (MSE): 0.70320 | Val Loss (MSE): 1.14063
Epoch 08/15 | Train Loss (MSE): 0.66803 | Val Loss (MSE): 1.15142
Epoch 09/15 | Train Loss (MSE): 0.64975 | Val Loss (MSE): 1.19111
Epoch 10/15 | Train Loss (MSE): 0.61491 | Val Loss (MSE): 1.23947
Epoch 11/15 | Train Loss (MSE): 0.59942 | Val Loss (MSE): 1.25987
Epoch 12/15 | Train Loss (MSE): 0.56662 | Val Loss (MSE): 1.38203
Epoch 13/15 | Train Loss (MSE): 0.53775 | Val Loss (MSE): 1.31230
Epoch 14/15 | Train Loss (MSE): 0.51721 | Val Loss (MSE): 1.42421
Epoch 15/15 | Train Loss (MSE): 0.48768 | Val Los

### Directional Signal Metrics

In [11]:
from sklearn.metrics import f1_score, classification_report

# 1. Convert continuous log returns to binary directional signals
# (Assuming 'true_celsius' and 'pred_celsius' variables hold your final unscaled returns)
actual_direction = (true_celsius > 0).astype(int)
predicted_direction = (pred_celsius > 0).astype(int)

# 2. Calculate the F1-Score
# F1 balances Precision (how many of our 'Up' predictions were right)
# and Recall (how many of the actual 'Up' days did we capture)
f1 = f1_score(actual_direction, predicted_direction)

print("\n=========================================")
print("       DIRECTIONAL SIGNAL METRICS        ")
print("=========================================")
print(f"F1-Score: {f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(actual_direction, predicted_direction, target_names=['Down (0)', 'Up (1)']))
print("=========================================")


       DIRECTIONAL SIGNAL METRICS        
F1-Score: 0.5732

Detailed Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.45      0.43      0.44        61
      Up (1)       0.56      0.58      0.57        77

    accuracy                           0.51       138
   macro avg       0.51      0.51      0.51       138
weighted avg       0.51      0.51      0.51       138



## LSTM

### architecture

In [12]:
# ==========================================
# 1. THE LSTM ARCHITECTURE
# ==========================================
class ForecastingLSTM(nn.Module):
    def __init__(self, input_features, hidden_dim):
        super().__init__()

        # batch_first=True expects input shape: [Batch Size, Sequence Length, Features]
        self.lstm = nn.LSTM(
            input_size=input_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        # Maps the final hidden memory state to a single continuous prediction
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: [32, 24, 13]
        lstm_out, (hidden_state, cell_state) = self.lstm(x)

        # Slice the tensor to grab the memory state of the very last hour in the sequence
        # lstm_out[:, -1, :] grabs the 24th time step for every item in the batch
        last_time_step = lstm_out[:, -1, :]

        predictions = self.linear(last_time_step)
        # predictions shape: [32, 1]

        return predictions

### training & Dianogsis

In [13]:
# ==========================================
# 2. HYPERPARAMETERS & SETUP
# ==========================================
epochs = 15
learning_rate = 0.0005 # LSTMs often benefit from a slightly lower LR than standard DNNs
weight_decay = 1e-5    # L2 regularization to help prevent overfitting

# Instantiate the model (13 features in, 64 hidden memory cells)
model = ForecastingLSTM(input_features=len(feature_cols), hidden_dim=64)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("Starting LSTM Training Loop...\n")

# ==========================================
# 3. THE TRAINING & VALIDATION LOOP
# ==========================================
for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = train_loss / len(train_dataloader.dataset)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_dataloader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = val_loss / len(val_dataloader.dataset)

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss (MSE): {epoch_train_loss:.5f} | Val Loss (MSE): {epoch_val_loss:.5f}")


# ==========================================
# 4. THE TESTING PHASE (UNSEEN DATA)
# ==========================================
print("\nTraining Complete. Evaluating LSTM on Test Set...")
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        predictions = model(X_batch)

        all_predictions.append(predictions.numpy())
        all_actuals.append(y_batch.numpy())

all_predictions = np.vstack(all_predictions)
all_actuals = np.vstack(all_actuals)

# Convert Scaled Z-Scores back to Degrees Celsius
true_celsius = target_scaler.inverse_transform(all_actuals)
pred_celsius = target_scaler.inverse_transform(all_predictions)

# Calculate Final Metrics
final_mae = mean_absolute_error(true_celsius, pred_celsius)
final_mse = mean_squared_error(true_celsius, pred_celsius)
final_rmse = np.sqrt(final_mse)

print("\n=========================================")
print("         FINAL LSTM TEST RESULTS         ")
print("=========================================")
print(f"Mean Absolute Error (MAE):    {final_mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f}")
print("=========================================")

Starting LSTM Training Loop...

Epoch 01/15 | Train Loss (MSE): 0.83372 | Val Loss (MSE): 1.11235
Epoch 02/15 | Train Loss (MSE): 0.82389 | Val Loss (MSE): 1.10350
Epoch 03/15 | Train Loss (MSE): 0.82219 | Val Loss (MSE): 1.10252
Epoch 04/15 | Train Loss (MSE): 0.82158 | Val Loss (MSE): 1.10353
Epoch 05/15 | Train Loss (MSE): 0.82081 | Val Loss (MSE): 1.10151
Epoch 06/15 | Train Loss (MSE): 0.81998 | Val Loss (MSE): 1.10155
Epoch 07/15 | Train Loss (MSE): 0.81954 | Val Loss (MSE): 1.10166
Epoch 08/15 | Train Loss (MSE): 0.81860 | Val Loss (MSE): 1.10248
Epoch 09/15 | Train Loss (MSE): 0.81857 | Val Loss (MSE): 1.10386
Epoch 10/15 | Train Loss (MSE): 0.81844 | Val Loss (MSE): 1.10560
Epoch 11/15 | Train Loss (MSE): 0.81725 | Val Loss (MSE): 1.10551
Epoch 12/15 | Train Loss (MSE): 0.81671 | Val Loss (MSE): 1.10588
Epoch 13/15 | Train Loss (MSE): 0.81643 | Val Loss (MSE): 1.10758
Epoch 14/15 | Train Loss (MSE): 0.81606 | Val Loss (MSE): 1.10797
Epoch 15/15 | Train Loss (MSE): 0.81636 | Va

### Directional Signal Metrics

In [14]:
from sklearn.metrics import f1_score, classification_report

# 1. Convert continuous log returns to binary directional signals
# (Assuming 'true_celsius' and 'pred_celsius' variables hold your final unscaled returns)
actual_direction = (true_celsius > 0).astype(int)
predicted_direction = (pred_celsius > 0).astype(int)

# 2. Calculate the F1-Score
# F1 balances Precision (how many of our 'Up' predictions were right)
# and Recall (how many of the actual 'Up' days did we capture)
f1 = f1_score(actual_direction, predicted_direction)

print("\n=========================================")
print("       DIRECTIONAL SIGNAL METRICS        ")
print("=========================================")
print(f"F1-Score: {f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(actual_direction, predicted_direction, target_names=['Down (0)', 'Up (1)']))
print("=========================================")


       DIRECTIONAL SIGNAL METRICS        
F1-Score: 0.6809

Detailed Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.52      0.23      0.32        61
      Up (1)       0.58      0.83      0.68        77

    accuracy                           0.57       138
   macro avg       0.55      0.53      0.50       138
weighted avg       0.55      0.57      0.52       138



## RNN

### architecture

In [15]:
import torch.nn as nn

class ForecastingRNN(nn.Module):
    def __init__(self, input_features, hidden_dim):
        super().__init__()

        # We swap nn.LSTM for nn.RNN
        self.rnn = nn.RNN(
            input_size=input_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # IMPORTANT DIFFERENCE: nn.RNN only outputs the sequence and a single hidden state
        # It does NOT output a (hidden_state, cell_state) tuple like an LSTM does.
        rnn_out, hidden_state = self.rnn(x)

        # We still slice the tensor to grab the 24th time step (the final hour)
        last_time_step = rnn_out[:, -1, :]

        predictions = self.linear(last_time_step)

        return predictions

### training & Dianogsis

In [16]:
# ==========================================
# 2. HYPERPARAMETERS & SETUP
# ==========================================
epochs = 15
learning_rate = 0.0005
weight_decay = 1e-5

# Swap to the Vanilla RNN architecture
model = ForecastingRNN(input_features=len(feature_cols), hidden_dim=64)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("Starting RNN Training Loop...\n")

# ==========================================
# 3. THE TRAINING & VALIDATION LOOP
# ==========================================
for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = train_loss / len(train_dataloader.dataset)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_dataloader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = val_loss / len(val_dataloader.dataset)

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss (MSE): {epoch_train_loss:.5f} | Val Loss (MSE): {epoch_val_loss:.5f}")


# ==========================================
# 4. THE TESTING PHASE (UNSEEN DATA)
# ==========================================
print("\nTraining Complete. Evaluating RNN on Test Set...")
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        predictions = model(X_batch)

        all_predictions.append(predictions.numpy())
        all_actuals.append(y_batch.numpy())

all_predictions = np.vstack(all_predictions)
all_actuals = np.vstack(all_actuals)

# Convert Scaled Z-Scores back to Degrees Celsius
true_celsius = target_scaler.inverse_transform(all_actuals)
pred_celsius = target_scaler.inverse_transform(all_predictions)

# Calculate Final Metrics
final_mae = mean_absolute_error(true_celsius, pred_celsius)
final_mse = mean_squared_error(true_celsius, pred_celsius)
final_rmse = np.sqrt(final_mse)

print("\n=========================================")
print("         FINAL RNN TEST RESULTS          ")
print("=========================================")
print(f"Mean Absolute Error (MAE):    {final_mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f}")
print("=========================================")

Starting RNN Training Loop...

Epoch 01/15 | Train Loss (MSE): 0.82591 | Val Loss (MSE): 1.09791
Epoch 02/15 | Train Loss (MSE): 0.82183 | Val Loss (MSE): 1.09146
Epoch 03/15 | Train Loss (MSE): 0.82108 | Val Loss (MSE): 1.09156
Epoch 04/15 | Train Loss (MSE): 0.81762 | Val Loss (MSE): 1.08945
Epoch 05/15 | Train Loss (MSE): 0.81797 | Val Loss (MSE): 1.08697
Epoch 06/15 | Train Loss (MSE): 0.81621 | Val Loss (MSE): 1.08835
Epoch 07/15 | Train Loss (MSE): 0.81380 | Val Loss (MSE): 1.08630
Epoch 08/15 | Train Loss (MSE): 0.81272 | Val Loss (MSE): 1.08959
Epoch 09/15 | Train Loss (MSE): 0.81250 | Val Loss (MSE): 1.09481
Epoch 10/15 | Train Loss (MSE): 0.81130 | Val Loss (MSE): 1.09123
Epoch 11/15 | Train Loss (MSE): 0.80963 | Val Loss (MSE): 1.08839
Epoch 12/15 | Train Loss (MSE): 0.81079 | Val Loss (MSE): 1.08920
Epoch 13/15 | Train Loss (MSE): 0.80910 | Val Loss (MSE): 1.09047
Epoch 14/15 | Train Loss (MSE): 0.80735 | Val Loss (MSE): 1.09008
Epoch 15/15 | Train Loss (MSE): 0.80890 | Val

### Directional Signal Metrics

In [17]:
from sklearn.metrics import f1_score, classification_report

# 1. Convert continuous log returns to binary directional signals
# (Assuming 'true_celsius' and 'pred_celsius' variables hold your final unscaled returns)
actual_direction = (true_celsius > 0).astype(int)
predicted_direction = (pred_celsius > 0).astype(int)

# 2. Calculate the F1-Score
# F1 balances Precision (how many of our 'Up' predictions were right)
# and Recall (how many of the actual 'Up' days did we capture)
f1 = f1_score(actual_direction, predicted_direction)

print("\n=========================================")
print("       DIRECTIONAL SIGNAL METRICS        ")
print("=========================================")
print(f"F1-Score: {f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(actual_direction, predicted_direction, target_names=['Down (0)', 'Up (1)']))
print("=========================================")


       DIRECTIONAL SIGNAL METRICS        
F1-Score: 0.6108

Detailed Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.46      0.36      0.40        61
      Up (1)       0.57      0.66      0.61        77

    accuracy                           0.53       138
   macro avg       0.51      0.51      0.51       138
weighted avg       0.52      0.53      0.52       138

